# Chapter 8: Orchestration Progression

**The 80/20 Rule of Data Orchestration**

This notebook demonstrates the progressive complexity ladder:

1. **Level 1: Make + cron** (80% of use cases)
2. **Level 2: Prefect** (15% of use cases)
3. **Level 3: Dagster** (4% of use cases)
4. **Level 4: Temporal** (1% of use cases)

**Rule:** Don't skip levels. Master each tier before adding complexity.

## Setup

In [ ]:
# Install dependencies
# !pip install polars duckdb prefect dagster dagster-polars dagster-duckdb pandera

In [ ]:
import polars as pl
import duckdb
from pathlib import Path
from datetime import datetime, timedelta

# Create data directories
for dir_name in ['data/raw', 'data/staging', 'data/curated', 'logs']:
    Path(dir_name).mkdir(parents=True, exist_ok=True)

print("[OK] Setup complete")

## Generate Sample Data

In [ ]:
# Generate synthetic events
import random
from datetime import datetime, timedelta

def generate_events(n=10000):
    """Generate synthetic event data."""
    events = []
    start_date = datetime(2024, 1, 1)
    
    for i in range(n):
        events.append({
            'user_id': random.randint(1, 1000),
            'event_type': random.choice(['click', 'view', 'purchase']),
            'timestamp': (start_date + timedelta(hours=random.randint(0, 30*24))).isoformat(),
            'amount': random.uniform(10, 1000) if random.random() > 0.7 else 0
        })
    
    return pl.DataFrame(events)

events_df = generate_events(10000)
events_df.write_json('data/raw/events.json')

print(f"[OK] Generated {len(events_df):,} events")
events_df.head()

## Level 1: Make + cron

### Why Start Here
- **Zero dependencies** - Already installed on Linux/Mac
- **Declarative** - Express dependencies as a graph
- **Fast feedback** - Run locally before scheduling
- **Version controlled** - Automation is just text files

### Makefile Pattern

In [ ]:
# Simulate Make pipeline in Python
from pathlib import Path
import subprocess

def should_rebuild(target: Path, dependencies: list[Path]) -> bool:
    """Check if target needs rebuilding (Make-style)."""
    if not target.exists():
        return True
    
    target_time = target.stat().st_mtime
    return any(dep.stat().st_mtime > target_time for dep in dependencies if dep.exists())

# Example: staging depends on raw
raw_file = Path('data/raw/events.json')
staging_file = Path('data/staging/events.parquet')

if should_rebuild(staging_file, [raw_file]):
    print(f"Rebuilding {staging_file}...")
    df = pl.read_json(raw_file)
    df.write_parquet(staging_file)
    print("[OK] Rebuilt")
else:
    print("[SKIP] Staging file is up to date")

### When to Graduate from Make + cron

Move to Level 2 when:
1. **Flaky dependencies** - API rate limits cause random failures
2. **Long pipelines** - Steps take >30 minutes
3. **Parallel execution** - Independent tasks could run concurrently
4. **State management** - Need to track "did this row process?"
5. **Observability** - `tail -f logs/*` isn't enough

## Level 2: Prefect

### Why Prefect
- **Retries with backoff** - Built-in retry logic for flaky steps
- **Caching** - Skip expensive computations if inputs haven't changed
- **Parallelism** - Run independent tasks concurrently
- **Local-first** - SQLite backend, no server required

In [ ]:
from prefect import flow, task
from prefect.tasks import task_input_hash
from datetime import timedelta

@task(retries=3, retry_delay_seconds=5)
def extract_data(source_path: str) -> str:
    """Extract data with automatic retries."""
    print(f"Extracting from {source_path}...")
    # Simulate flaky operation
    import random
    if random.random() < 0.3:  # 30% failure rate
        raise Exception("Simulated API failure")
    return source_path

@task(cache_key_fn=task_input_hash, cache_expiration=timedelta(hours=1))
def transform_data(input_path: str) -> str:
    """Transform with result caching."""
    print(f"Transforming {input_path}...")
    df = pl.read_json(input_path)
    
    # Clean data
    df = df.filter(pl.col("user_id").is_not_null())
    df = df.with_columns(pl.col("timestamp").str.to_datetime())
    
    output_path = input_path.replace('raw', 'staging').replace('.json', '.parquet')
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    df.write_parquet(output_path)
    
    return output_path

@task
def compute_metrics(staging_path: str) -> str:
    """Run DuckDB aggregations."""
    print(f"Computing metrics from {staging_path}...")
    conn = duckdb.connect()
    
    query = f"""
    COPY (
        SELECT
            DATE_TRUNC('day', timestamp) as date,
            COUNT(DISTINCT user_id) as dau,
            COUNT(*) as events
        FROM read_parquet('{staging_path}')
        GROUP BY 1
    ) TO 'data/curated/daily_metrics.parquet' (FORMAT PARQUET)
    """
    
    conn.execute(query)
    return 'data/curated/daily_metrics.parquet'

@flow(name="prefect-demo", log_prints=True)
def prefect_pipeline():
    """Main Prefect pipeline."""
    raw_path = extract_data('data/raw/events.json')
    staging_path = transform_data(raw_path)
    metrics_path = compute_metrics(staging_path)
    print(f"[OK] Pipeline complete. Metrics at {metrics_path}")

# Run the flow
prefect_pipeline()

### Prefect Parallel Execution

In [ ]:
@flow
def parallel_flow():
    """Run multiple extracts in parallel."""
    # Submit tasks for parallel execution
    futures = []
    sources = ['events', 'users', 'products']
    
    for source in sources:
        future = extract_data.submit(f'data/raw/{source}.json')
        futures.append(future)
    
    # Wait for all to complete
    results = [f.result() for f in futures]
    print(f"[OK] Extracted {len(results)} sources in parallel")
    return results

# parallel_flow()

## Level 3: Dagster

### Mental Model Shift

**Prefect thinks in tasks:**
```
extract() -> transform() -> load()
```

**Dagster thinks in assets:**
```
raw_events -> cleaned_events -> daily_metrics
     |              |                |
 (exists?)      (valid?)        (complete?)
```

In [ ]:
from dagster import asset, AssetExecutionContext, Definitions

@asset(
    group_name="ingestion",
    compute_kind="polars"
)
def raw_events_asset(context: AssetExecutionContext) -> pl.DataFrame:
    """Raw events from source."""
    df = pl.read_json('data/raw/events.json')
    context.log.info(f"Loaded {len(df):,} raw events")
    return df

@asset(
    deps=[raw_events_asset],
    compute_kind="polars"
)
def cleaned_events_asset(context: AssetExecutionContext, raw_events_asset: pl.DataFrame) -> pl.DataFrame:
    """Cleaned and validated events."""
    df = (
        raw_events_asset
        .filter(pl.col("user_id").is_not_null())
        .filter(pl.col("event_type").is_in(["click", "view", "purchase"]))
        .with_columns(pl.col("timestamp").str.to_datetime())
    )
    
    output_path = Path('data/staging/events.parquet')
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.write_parquet(output_path)
    
    context.log.info(f"Cleaned {len(df):,} events")
    return df

@asset(
    deps=[cleaned_events_asset],
    compute_kind="duckdb"
)
def daily_metrics_asset(context: AssetExecutionContext) -> None:
    """Daily aggregated metrics."""
    conn = duckdb.connect()
    
    conn.execute("""
        COPY (
            SELECT
                DATE_TRUNC('day', timestamp) as date,
                event_type,
                COUNT(*) as event_count,
                COUNT(DISTINCT user_id) as unique_users
            FROM read_parquet('data/staging/events.parquet')
            GROUP BY 1, 2
        ) TO 'data/curated/daily_metrics.parquet' (FORMAT PARQUET)
    """)
    
    context.log.info("Metrics computed")

# Define assets
defs = Definitions(
    assets=[raw_events_asset, cleaned_events_asset, daily_metrics_asset]
)

print("[OK] Dagster assets defined")
print("Run with: dagster asset materialize --select '*'")

### Type-Safe Assets with Pandera

In [ ]:
import pandera.polars as pa

class EventSchema(pa.DataFrameModel):
    """Schema for cleaned events."""
    user_id: int = pa.Field(ge=1)
    event_type: str = pa.Field(isin=["click", "view", "purchase"])
    timestamp: pl.Datetime
    amount: float = pa.Field(ge=0.0)

    class Config:
        strict = True
        coerce = True

@asset
def validated_events(context: AssetExecutionContext) -> pl.DataFrame:
    """Events with schema validation."""
    df = pl.read_json('data/raw/events.json')
    
    # Transform
    df = (
        df
        .filter(pl.col("user_id").is_not_null())
        .with_columns(pl.col("timestamp").str.to_datetime())
    )
    
    # Validate against schema
    try:
        validated_df = EventSchema.validate(df)
        context.log.info(f"[OK] Schema validation passed for {len(validated_df):,} rows")
        return validated_df
    except pa.errors.SchemaErrors as e:
        context.log.error(f"Schema validation failed: {e.failure_cases}")
        raise

print("[OK] Type-safe asset defined")

## Decision Matrix

| Requirement          | Make+cron | Prefect  | Dagster  | Temporal |
|---------------------|-----------|----------|----------|-----------|
| Zero dependencies    | ✓         |          |          |           |
| Retries with backoff |           | ✓        | ✓        | ✓         |
| Parallel execution   |           | ✓        | ✓        | ✓         |
| Type-safe schemas    |           |          | ✓        | ✓         |
| Asset-centric view   |           |          | ✓        |           |
| Durability (crash)   |           |          |          | ✓         |
| Learning curve       | 1 day     | 3 days   | 1 week   | 2 weeks   |

## When to Use Each Level

### Start with Make + cron if:
- Pipeline runs daily or less frequently
- Steps complete in minutes, not hours
- Failures are obvious (you check logs)
- No external API rate limits to manage

### Add Prefect when:
- API rate limits cause random failures
- Steps take >30 minutes
- You need parallel execution
- You want result caching

### Add Dagster when:
- Type safety matters
- You care about "what data exists" vs "what tasks ran"
- Upstream/downstream teams need formal interfaces
- You want to test pipelines without running them

### Add Temporal when:
- Workflows take hours or days
- Need to pause for approval
- Must survive crashes and resume
- **Spoiler: You probably don't need this**

## Verify Results

In [ ]:
# Check curated metrics
if Path('data/curated/daily_metrics.parquet').exists():
    metrics = pl.read_parquet('data/curated/daily_metrics.parquet')
    print(f"\n[OK] Daily metrics computed: {len(metrics)} days\n")
    print(metrics.head(10))
else:
    print("[INFO] Run a pipeline first to generate metrics")

## Key Takeaways

1. **Start simple** - Make + cron covers 80% of use cases
2. **Graduate when pain is real** - Don't over-engineer early
3. **Prefect for retries** - When APIs are flaky
4. **Dagster for contracts** - When type safety matters
5. **Temporal for durability** - When workflows take days (rare)
6. **Most projects stop at Dagster** - And that's perfectly fine

**Remember:** Boring tech that works beats shiny tech that's complex.